# Stock Market Analysis and LSTM-Based Price Prediction

Explore historical data for four technology stocks, analyze returns and correlations, and evaluate an LSTM model for Apple closing-price prediction.

This is an educational project. It is not financial advice and does not provide a validated trading strategy.


Time Series data is a series of data points indexed in time order. Time series data is everywhere, so manipulating them is important for any data analyst or data scientist.

In this notebook, we will discover and explore data from the stock market, particularly some technology stocks (Apple, Amazon, Google, and Microsoft). We will learn how to use yfinance to get stock information, and visualize different aspects of it using Seaborn and Matplotlib. we will look at a few ways of analyzing the risk of a stock, based on its previous performance history. We will also be predicting future stock prices through a Long Short Term Memory (LSTM) method!

We'll be answering the following questions along the way:

    1.) What was the change in price of the stock over time?
    2.) What was the daily return of the stock on average?
    3.) What was the moving average of the various stocks?
    4.) What was the correlation between different stocks'?
    5.) How much value do we put at risk by investing in a particular stock?
    6.) How can we attempt to predict future stock behavior? (Predicting the closing price stock price of APPLE inc using LSTM)

***   

## Getting the Data
The first step is to get the data and load it to memory. We will get our stock data from the Yahoo Finance website. Yahoo Finance is a rich resource of financial market data and tools to find compelling investments. To get the data from Yahoo Finance, we will be using yfinance library which offers a threaded and Pythonic way to download market data from Yahoo. Check this article to learn more about yfinance: [Reliably download historical market data from with Python](https://aroussi.com/post/python-yahoo-finance)

# 1. What was the change in price of the stock overtime?

In this section we'll go over how to handle requesting stock information with pandas, and how to analyze basic attributes of a stock.

In [ ]:
# Install project dependencies in your active Python environment before running the notebook.
# See requirements.txt in this project directory.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
from datetime import datetime, timedelta

sns.set_style("whitegrid")
plt.style.use("fivethirtyeight")
%matplotlib inline

# Analyze four large technology companies over the most recent year.
tech_list = ["AAPL", "GOOG", "MSFT", "AMZN"]
end = datetime.now()
start = end - timedelta(days=365)

company_names = {
    "AAPL": "APPLE",
    "GOOG": "GOOGLE",
    "MSFT": "MICROSOFT",
    "AMZN": "AMAZON",
}

company_list = []
for ticker in tech_list:
    company = yf.download(
        ticker,
        start=start,
        end=end,
        auto_adjust=False,
        progress=False,
    )
    if company.empty:
        raise RuntimeError(f"No market data was returned for {ticker}. Check your internet connection or ticker symbol.")
    # Recent yfinance versions may return MultiIndex columns for single tickers.
    if isinstance(company.columns, pd.MultiIndex):
        company.columns = company.columns.get_level_values(0)
    company["company_name"] = company_names[ticker]
    company_list.append(company)
    globals()[ticker] = company

df = pd.concat(company_list, axis=0)
df.tail(10)


Reviewing the content of our data, we can see that the data is numeric and the date is the index of the data. Notice also that weekends are missing from the records.

**Quick note:** Using `globals()` is a sloppy way of setting the `DataFrame` names, but it's simple. Now we have our data, let's perform some basic data analysis and check our data.

## Descriptive Statistics about the Data
`.describe()` generates descriptive statistics. Descriptive statistics include those that summarize the central tendency, dispersion, and shape of a dataset’s distribution, excluding `NaN` values.

Analyzes both numeric and object series, as well as `DataFrame` column sets of mixed data types. The output will vary depending on what is provided. Refer to the notes below for more detail.

In [ ]:
# Summary Stats
AAPL.describe()

We have only 255 records in one year because weekends are not included in the data.

## Information About the Data
`.info()` method prints information about a DataFrame including the index `dtype` and columns, non-null values, and memory usage.

In [ ]:
# General info
AAPL.info()

## Closing Price

The closing price is the last price at which the stock is traded during the regular trading day. A stock’s closing price is the standard benchmark used by investors to track its performance over time.

In [ ]:
# Let's see a historical view of the closing price
plt.figure(figsize=(15, 10))
plt.subplots_adjust(top=1.25, bottom=1.2)

for i, company in enumerate(company_list, 1):
    plt.subplot(2, 2, i)
    company['Adj Close'].plot()
    plt.ylabel('Adj Close')
    plt.xlabel(None)
    plt.title(f"Closing Price of {tech_list[i - 1]}")
    
plt.tight_layout()

## Volume of Sales
Volume is the amount of an asset or security that changes hands over some period of time, often over the course of a day. For instance, the stock trading volume would refer to the number of shares of security traded between its daily open and close. Trading volume, and changes to volume over the course of time, are important inputs for technical traders.

In [ ]:
# Now let's plot the total volume of stock being traded each day
plt.figure(figsize=(15, 10))
plt.subplots_adjust(top=1.25, bottom=1.2)

for i, company in enumerate(company_list, 1):
    plt.subplot(2, 2, i)
    company['Volume'].plot()
    plt.ylabel('Volume')
    plt.xlabel(None)
    plt.title(f"Sales Volume for {tech_list[i - 1]}")
    
plt.tight_layout()

Now that we've seen the visualizations for the closing price and the volume traded each day, let's go ahead and caculate the moving average for the stock.

# 2. What was the moving average of the various stocks?

The moving average (MA) is a simple technical analysis tool that smooths out price data by creating a constantly updated average price. The average is taken over a specific period of time, like 10 days, 20 minutes, 30 weeks, or any time period the trader chooses.

In [ ]:
ma_day = [10, 20, 50]

for ma in ma_day:
    for company in company_list:
        column_name = f"MA for {ma} days"
        company[column_name] = company['Adj Close'].rolling(ma).mean()
        

fig, axes = plt.subplots(nrows=2, ncols=2)
fig.set_figheight(10)
fig.set_figwidth(15)

AAPL[['Adj Close', 'MA for 10 days', 'MA for 20 days', 'MA for 50 days']].plot(ax=axes[0,0])
axes[0,0].set_title('APPLE')

GOOG[['Adj Close', 'MA for 10 days', 'MA for 20 days', 'MA for 50 days']].plot(ax=axes[0,1])
axes[0,1].set_title('GOOGLE')

MSFT[['Adj Close', 'MA for 10 days', 'MA for 20 days', 'MA for 50 days']].plot(ax=axes[1,0])
axes[1,0].set_title('MICROSOFT')

AMZN[['Adj Close', 'MA for 10 days', 'MA for 20 days', 'MA for 50 days']].plot(ax=axes[1,1])
axes[1,1].set_title('AMAZON')

fig.tight_layout()

We see in the graph that the best values to measure the moving average are 10 and 20 days because we still capture trends in the data without noise.

# 3. What was the daily return of the stock on average?

Now that we've done some baseline analysis, let's go ahead and dive a little deeper. We're now going to analyze the risk of the stock. In order to do so we'll need to take a closer look at the daily changes of the stock, and not just its absolute value. Let's go ahead and use pandas to retrieve teh daily returns for the Apple stock.

In [ ]:
# We'll use pct_change to find the percent change for each day
for company in company_list:
    company['Daily Return'] = company['Adj Close'].pct_change()

# Then we'll plot the daily return percentage
fig, axes = plt.subplots(nrows=2, ncols=2)
fig.set_figheight(10)
fig.set_figwidth(15)

AAPL['Daily Return'].plot(ax=axes[0,0], legend=True, linestyle='--', marker='o')
axes[0,0].set_title('APPLE')

GOOG['Daily Return'].plot(ax=axes[0,1], legend=True, linestyle='--', marker='o')
axes[0,1].set_title('GOOGLE')

MSFT['Daily Return'].plot(ax=axes[1,0], legend=True, linestyle='--', marker='o')
axes[1,0].set_title('MICROSOFT')

AMZN['Daily Return'].plot(ax=axes[1,1], legend=True, linestyle='--', marker='o')
axes[1,1].set_title('AMAZON')

fig.tight_layout()

Great, now let's get an overall look at the average daily return using a histogram. We'll use seaborn to create both a histogram and kde plot on the same figure.

In [ ]:
plt.figure(figsize=(12, 9))

for i, company in enumerate(company_list, 1):
    plt.subplot(2, 2, i)
    company['Daily Return'].hist(bins=50)
    plt.xlabel('Daily Return')
    plt.ylabel('Counts')
    plt.title(f'{company_name[i - 1]}')
    
plt.tight_layout()

# 4. What was the correlation between different stocks closing prices?

Correlation is a statistic that measures the degree to which two variables move in relation to each other which has a value that must fall between -1.0 and +1.0. Correlation measures association, but doesn’t show if x causes y or vice versa — or if the association is caused by a third factor[1].

Now what if we wanted to analyze the returns of all the stocks in our list? Let's go ahead and build a DataFrame with all the ['Close'] columns for each of the stocks dataframes.

In [ ]:
# Download adjusted closing prices for all tickers in one request.
closing_df = yf.download(
    tech_list,
    start=start,
    end=end,
    auto_adjust=False,
    progress=False,
)["Adj Close"]

if isinstance(closing_df, pd.Series):
    closing_df = closing_df.to_frame(name=tech_list[0])

tech_rets = closing_df.pct_change()
tech_rets.head()


Now we can compare the daily percentage return of two stocks to check how correlated. First let's see a sotck compared to itself.

In [ ]:
# Comparing Google to itself should show a perfectly linear relationship
sns.jointplot(x='GOOG', y='GOOG', data=tech_rets, kind='scatter', color='seagreen')

In [ ]:
# We'll use joinplot to compare the daily returns of Google and Microsoft
sns.jointplot(x='GOOG', y='MSFT', data=tech_rets, kind='scatter')

So now we can see that if two stocks are perfectly (and positivley) correlated with each other a linear relationship bewteen its daily return values should occur. 

Seaborn and pandas make it very easy to repeat this comparison analysis for every possible combination of stocks in our technology stock ticker list. We can use sns.pairplot() to automatically create this plot

In [ ]:
# We can simply call pairplot on our DataFrame for an automatic visual analysis 
# of all the comparisons

sns.pairplot(tech_rets, kind='reg')

Above we can see all the relationships on daily returns between all the stocks. A quick glance shows an interesting correlation between Google and Amazon daily returns. It might be interesting to investigate that individual comaprison. 

While the simplicity of just calling `sns.pairplot()` is fantastic we can also use `sns.PairGrid()` for full control of the figure, including what kind of plots go in the diagonal, the upper triangle, and the lower triangle. Below is an example of utilizing the full power of seaborn to achieve this result.

In [ ]:
# Set up our figure by naming it returns_fig, call PairPLot on the DataFrame
return_fig = sns.PairGrid(tech_rets.dropna())

# Using map_upper we can specify what the upper triangle will look like.
return_fig.map_upper(plt.scatter, color='purple')

# We can also define the lower triangle in the figure, inclufing the plot type (kde) 
# or the color map (BluePurple)
return_fig.map_lower(sns.kdeplot, cmap='cool_d')

# Finally we'll define the diagonal as a series of histogram plots of the daily return
return_fig.map_diag(plt.hist, bins=30)

In [ ]:
# Set up our figure by naming it returns_fig, call PairPLot on the DataFrame
returns_fig = sns.PairGrid(closing_df)

# Using map_upper we can specify what the upper triangle will look like.
returns_fig.map_upper(plt.scatter,color='purple')

# We can also define the lower triangle in the figure, inclufing the plot type (kde) or the color map (BluePurple)
returns_fig.map_lower(sns.kdeplot,cmap='cool_d')

# Finally we'll define the diagonal as a series of histogram plots of the daily return
returns_fig.map_diag(plt.hist,bins=30)

Finally, we could also do a correlation plot, to get actual numerical values for the correlation between the stocks' daily return values. By comparing the closing prices, we see an interesting relationship between Microsoft and Apple.

In [ ]:
plt.figure(figsize=(12, 10))

plt.subplot(2, 2, 1)
sns.heatmap(tech_rets.corr(), annot=True, cmap='summer')
plt.title('Correlation of stock return')

plt.subplot(2, 2, 2)
sns.heatmap(closing_df.corr(), annot=True, cmap='summer')
plt.title('Correlation of stock closing price')

Just like we suspected in our `PairPlot` we see here numerically and visually that Microsoft and Amazon had the strongest correlation of daily stock return. It's also interesting to see that all the technology comapnies are positively correlated.

# 5. How much value do we put at risk by investing in a particular stock?

There are many ways we can quantify risk, one of the most basic ways using the information we've gathered on daily percentage returns is by comparing the expected return with the standard deviation of the daily returns.

In [ ]:
rets = tech_rets.dropna()

area = np.pi * 20

plt.figure(figsize=(10, 8))
plt.scatter(rets.mean(), rets.std(), s=area)
plt.xlabel('Expected return')
plt.ylabel('Risk')

for label, x, y in zip(rets.columns, rets.mean(), rets.std()):
    plt.annotate(label, xy=(x, y), xytext=(50, 50), textcoords='offset points', ha='right', va='bottom', 
                 arrowprops=dict(arrowstyle='-', color='blue', connectionstyle='arc3,rad=-0.3'))

# 6. Predicting the closing price stock price of APPLE inc:

In [ ]:
# Download Apple's historical prices for the LSTM example.
df = yf.download(
    "AAPL",
    start="2012-01-01",
    end=datetime.now(),
    auto_adjust=False,
    progress=False,
)

if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

if df.empty or "Close" not in df.columns:
    raise RuntimeError("Apple price data could not be downloaded or the Close column is missing.")

df.tail()


In [ ]:
plt.figure(figsize=(16,6))
plt.title('Close Price History')
plt.plot(df['Close'])
plt.xlabel('Date', fontsize=18)
plt.ylabel('Close Price USD ($)', fontsize=18)
plt.show()

In [ ]:
# Use the closing price as the target series.
data = df[["Close"]].dropna().copy()
dataset = data.to_numpy(dtype=float)
training_data_len = int(len(dataset) * 0.95)

if training_data_len <= 60 or len(dataset) - training_data_len == 0:
    raise ValueError("Not enough observations to create 60-day sequences and a test set.")

training_data_len


In [ ]:
# Fit the scaler on training data only to avoid leaking test-period information.
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler(feature_range=(0, 1))
scaler.fit(dataset[:training_data_len])
scaled_data = scaler.transform(dataset)

scaled_data[:5]


In [ ]:
# Create training sequences using the previous 60 trading days to predict the next close.
train_data = scaled_data[:training_data_len]
x_train = []
y_train = []

for i in range(60, len(train_data)):
    x_train.append(train_data[i - 60:i, 0])
    y_train.append(train_data[i, 0])

x_train = np.asarray(x_train, dtype=np.float32)
y_train = np.asarray(y_train, dtype=np.float32)
x_train = x_train.reshape((x_train.shape[0], x_train.shape[1], 1))

print("X train shape:", x_train.shape)
print("y train shape:", y_train.shape)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Input
from tensorflow.keras.callbacks import EarlyStopping

model = Sequential([
    Input(shape=(x_train.shape[1], 1)),
    LSTM(64, return_sequences=True),
    LSTM(32),
    Dense(16, activation="relu"),
    Dense(1),
])

model.compile(optimizer="adam", loss="mean_squared_error")

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
)

history = model.fit(
    x_train,
    y_train,
    validation_split=0.1,
    batch_size=32,
    epochs=30,
    callbacks=[early_stopping],
    shuffle=False,
    verbose=1,
)


In [ ]:
# Build test sequences with the preceding 60 observations as context.
test_data = scaled_data[training_data_len - 60:]
x_test = []
y_test = dataset[training_data_len:]

for i in range(60, len(test_data)):
    x_test.append(test_data[i - 60:i, 0])

x_test = np.asarray(x_test, dtype=np.float32).reshape((-1, 60, 1))

predictions_scaled = model.predict(x_test, verbose=0)
predictions = scaler.inverse_transform(predictions_scaled)

from sklearn.metrics import mean_absolute_error, mean_squared_error

rmse = float(np.sqrt(mean_squared_error(y_test, predictions)))
mae = float(mean_absolute_error(y_test, predictions))

# A simple persistence baseline predicts tomorrow's close as today's close.
# The first test prediction uses the final observed close immediately before the test period.
baseline_predictions = dataset[training_data_len - 1:-1]
baseline_rmse = float(np.sqrt(mean_squared_error(y_test, baseline_predictions)))
baseline_mae = float(mean_absolute_error(y_test, baseline_predictions))

metrics = pd.DataFrame({
    "Model": ["LSTM", "Persistence baseline"],
    "RMSE": [rmse, baseline_rmse],
    "MAE": [mae, baseline_mae],
})
metrics


In [ ]:
# Compare predictions with actual closing prices in the held-out test period.
train = data.iloc[:training_data_len].copy()
valid = data.iloc[training_data_len:].copy()
valid["LSTM Prediction"] = predictions.ravel()
valid["Persistence Baseline"] = baseline_predictions.ravel()

plt.figure(figsize=(16, 6))
plt.title("Apple Closing Price: Test-Period Predictions")
plt.xlabel("Date")
plt.ylabel("Close Price (USD)")
plt.plot(train.index, train["Close"], label="Training data")
plt.plot(valid.index, valid["Close"], label="Actual test close")
plt.plot(valid.index, valid["LSTM Prediction"], label="LSTM prediction")
plt.plot(valid.index, valid["Persistence Baseline"], label="Persistence baseline", alpha=0.7)
plt.legend(loc="best")
plt.tight_layout()
plt.show()


In [ ]:
# View actual values, LSTM predictions, and baseline predictions for the test period.
valid.tail(10)


# Summary

This notebook explores historical market data for AAPL, GOOG, MSFT, and AMZN, including closing prices, trading volume, moving averages, daily returns, return correlations, and a simple risk-return comparison.

It then trains an LSTM model to estimate Apple closing prices over a held-out historical test period. The model is evaluated with RMSE and MAE and compared with a persistence baseline that uses the previous closing price.

**Important limitations:** this is an educational time-series example, not financial advice or a ready-to-use trading strategy. Historical test-period predictions do not guarantee future performance. Results depend on the downloaded data and training run, and the evaluation does not include transaction costs or a trading simulation.

References:
- https://www.investopedia.com/terms/c/correlation.asp
- https://ranaroussi.github.io/yfinance/
